# Encoder, decoder и адаптация трансформера

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Различать encoder и decoder и выбирать объём адаптации под доступные данные и бюджет.

## Внимание внутри блока

Трансформер соединяет внимание с преобразованием каждой позиции, остаточными связями и нормализацией. **Остаточная связь** добавляет вход блока к его преобразованному результату; формы должны совпадать. Она помогает передавать сигнал через много слоёв. Несколько голов внимания используют разные наборы Q,K,V и затем объединяются.

Encoder обычно строит представление входной последовательности с доступом к обеим сторонам. Авторегрессионный decoder использует причинную маску для генерации следующего токена. В encoder-decoder модели decoder также обращается к выходам encoder через cross-attention: запросы из генерируемой части, ключи и значения из входной.

Остаточную связь можно проверить без библиотеки. Если вход позиции x=[1,2], а блок вычислил F(x)=[0.5,−1], результат x+F(x)=[1.5,1]. При двух головах каждая может вернуть своё описание позиции; их координаты соединяют и обучаемым линейным преобразованием возвращают размер d_model. Поэтому d_model в библиотечном слое должен делиться на nhead: в примере 4 признака разделены на две головы по 2.

## Разобранный пример: разные задачи

Для классификации готового сообщения допустимо видеть весь текст. Для предсказания его третьего слова нельзя использовать само третье и последующие слова. Одинаковая архитектурная деталь внимания выполняет разные задачи в зависимости от маски, входов и цели обучения.

Адаптация может обучать только новую голову, часть слоёв или все веса. Меньше обновляемых параметров обычно снижает память оптимизатора и риск подстройки на малых данных, но не гарантирует лучшее качество. Контекстное окно ограничивает число доступных токенов; обрезание может удалить важное условие задачи.

## Маленький вычислительный пример

В ноутбуке не скачиваем готовый трансформер. Обученное ранее представление не имитируем: используем случайный маленький nn.TransformerEncoderLayer только для проверки формы прямого прохода. Параметр batch_first=True задаёт порядок (объекты,токены,признаки), d_model — размер признака, nhead — число голов. Dropout=0 делает пример сравнимым; это не оценка понимания текста.

Для практического выбора запишите число размеченных примеров, память, задержку и допустимый способ использования внешних моделей. Простой текстовый baseline должен оставаться точкой сравнения.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(17)
block = nn.TransformerEncoderLayer(d_model=4, nhead=2, dim_feedforward=8, dropout=0., batch_first=True)
block.eval()
x = torch.arange(40, dtype=torch.float32).reshape(2, 5, 4) / 40
with torch.no_grad():
    output = block(x)
assert output.shape == x.shape
assert not output.requires_grad
print(output.shape)


## Типичные ошибки

Не называйте случайный блок предобученной LLM. Не путайте encoder-decoder и одну авторегрессионную сеть.

## Задачи

### 1. Базовая

Может ли encoder для классификации видеть весь готовый текст?

<details>
<summary>Подсказка</summary>

Будущее относительно генерации здесь не требуется.

</details>



### 2. Базовая

Зачем decoder причинная маска?

<details>
<summary>Подсказка</summary>

Следующий токен ещё неизвестен.

</details>



### 3. Прикладная

Вход (2,5,4) при batch_first=True: что значит 5?

<details>
<summary>Подсказка</summary>

Ось после объектов.

</details>



### 4. Прикладная

Можно ли сложить остаточную связь форм (2,5,4) и (2,5,8) напрямую?

<details>
<summary>Подсказка</summary>

Последние размеры различаются.

</details>



### 5. Олимпиадная

Почему обучение только головы может не решить новую задачу?

<details>
<summary>Подсказка</summary>

Представление может не сохранять нужный сигнал.

</details>



### 6. Олимпиадная

Что нарушится, если при следующем токене дать доступ к полному ответу?

<details>
<summary>Подсказка</summary>

Правильная информация доступна напрямую.

</details>



## Итоги и самопроверка

Выберите маску для классификации и продолжения текста. Чем отличаются стоимость и возможности обучения головы и всех слоёв?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Да, если весь текст доступен в момент классификации.

### Задача 2

Для исключения доступа к будущим правильным токенам.

### Задача 3

Пять токенов в каждой последовательности.

### Задача 4

Нет. Нужна согласованная размерность или отдельная проекция.

### Задача 5

Замороженные слои не могут восстановить потерянные различия, поэтому нужна проверка вариантов адаптации.

### Задача 6

Обучение и проверка станут нечестными относительно реальной генерации.